In [20]:
import os
import cv2
import gc
import numpy as np
import pandas as pd
import mediapipe as mp
from tqdm import tqdm

# ==========================================
# 1. PATH CONFIGURATION (UPDATE THESE!)
# ==========================================
# MST-E CONFIG
MST_E_ROOT = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\MonkSkinToneDataset\mst-e_data'
MST_E_CSV_RAW = os.path.join(MST_E_ROOT, 'mst-e_image_details.csv')
MST_E_OUTPUT_DIR = r'G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE'

# FACET CONFIG
FACET_ROOT_IMAGES = r'G:\Thesis\FACET_Dataset\Images' 
FACET_CSV_RAW = r'G:\Thesis\FACET_Dataset\Annotations\annotations\annotations.csv'
FACET_OUTPUT_DIR = r'G:\Thesis\FACET_Dataset\Segmented_FACET_0.2'

# Create Output Dirs
os.makedirs(MST_E_OUTPUT_DIR, exist_ok=True)
os.makedirs(FACET_OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. SEGMENTATION ENGINE
# ==========================================
def segment_face(image, face_mesh):
    """
    Accepts loaded image, returns cropped face with black background.
    """
    h_img, w_img, _ = image.shape
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_image)

    if not results.multi_face_landmarks:
        return None 

    landmarks = results.multi_face_landmarks[0]
    
    # Get Geometry
    points = np.array([(int(lm.x * w_img), int(lm.y * h_img)) for lm in landmarks.landmark])
    x, y, w_box, h_box = cv2.boundingRect(points)
    
    # Filter: Too Small or Weird Aspect Ratio
    if w_box < 50 or h_box < 50: return None
    aspect = h_box / w_box
    if aspect < 0.6 or aspect > 2.5: return None

    # Masking
    mask = np.zeros((h_img, w_img), dtype=np.uint8)
    hull = cv2.convexHull(points)
    cv2.fillConvexPoly(mask, hull, 255)
    segmented = cv2.bitwise_and(image, image, mask=mask)

    # Cropping with Padding
    pad = 10
    x = max(0, x - pad)
    y = max(0, y - pad)
    w_box = min(w_img, x + w_box + 2*pad) - x
    h_box = min(h_img, y + h_box + 2*pad) - y
    
    return segmented[y:y+h_box, x:x+w_box]

# ==========================================
# 3. PROCESS MST-E
# ==========================================
def process_mste():
    print("\n--- Processing MST-E Dataset ---")
    df = pd.read_csv(MST_E_CSV_RAW)
    df = df.rename(columns={'MST': 'mst_label'})
    
    valid_paths = []
    
    with mp.solutions.face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5) as face_mesh:
        
        for i, row in tqdm(enumerate(df.itertuples(index=False)), total=len(df)):
            # Handle Nested Paths: subject_X/image.jpg
            sub_folder = str(row.subject_name)
            fname = str(row.image_ID)
            
            input_path = os.path.join(MST_E_ROOT, sub_folder, fname)
            out_name = f"{sub_folder}_{fname}"
            output_path = os.path.join(MST_E_OUTPUT_DIR, out_name)
            
            if os.path.exists(output_path):
                valid_paths.append(output_path); continue
            
            if not os.path.exists(input_path):
                valid_paths.append(None); continue
                
            try:
                img = cv2.imread(input_path)
                if img is None: valid_paths.append(None); continue
                
                crop = segment_face(img, face_mesh)
                if crop is not None:
                    cv2.imwrite(output_path, crop)
                    valid_paths.append(output_path)
                else:
                    valid_paths.append(None)
            except:
                valid_paths.append(None)
                
            if i % 200 == 0: gc.collect()

    df['segmented_path'] = valid_paths
    df_clean = df.dropna(subset=['segmented_path'])
    out_csv = os.path.join(MST_E_OUTPUT_DIR, 'mst_e_final.csv')
    df_clean.to_csv(out_csv, index=False)
    print(f"MST-E Done. Saved {len(df_clean)} images.")
    return out_csv

# ==========================================
# 4. PROCESS FACET
# ==========================================
def process_facet(consensus_threshold=0.5):
    print("\n--- Processing FACET Dataset ---")
    # Load and filter FACET logic
    df = pd.read_csv(FACET_CSV_RAW)
    skin_cols = [c for c in df.columns if c.startswith("skin_tone_")]
    
    # Filter valid labels
    df = df[df[skin_cols].sum(axis=1) > 0] # Must have votes

    # Calculate Consensus Agreement
    # Find the max votes for the winning class
    max_votes = df[skin_cols].max(axis=1)
    # Find total votes cast
    total_votes = df[skin_cols].sum(axis=1)
    
    # Calculate Agreement Ratio (e.g., 8/10 = 0.8)
    agreement_ratio = max_votes / total_votes
    
    # --- FILTERING STEP ---
    # Only keep rows where agreement >= threshold
    initial_len = len(df)
    df = df[agreement_ratio >= consensus_threshold]
    print(f"Consensus Filter: Dropped {initial_len - len(df)} images with low annotator agreement (<{consensus_threshold*100}%).")
    
    # 4. Assign Final Label
    df["mst_label"] = df[skin_cols].idxmax(axis=1).str.replace("skin_tone_", "")
    
    # 5. Clean up 'na' and convert to int
    df = df[df['mst_label'] != "na"]
    df['mst_label'] = df['mst_label'].astype(int)

    valid_paths = []
    
    with mp.solutions.face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5) as face_mesh:
        
        # Iterating
        for i, row in tqdm(enumerate(df.itertuples(index=False)), total=len(df)):
            fname = row.filename # FACET usually has 'filename' or 'image_id'
            input_path = os.path.join(FACET_ROOT_IMAGES, fname)
            output_path = os.path.join(FACET_OUTPUT_DIR, fname)
            
            if os.path.exists(output_path):
                valid_paths.append(output_path); continue
            
            if not os.path.exists(input_path):
                valid_paths.append(None); continue

            try:
                img = cv2.imread(input_path)
                if img is None: valid_paths.append(None); continue
                
                crop = segment_face(img, face_mesh)
                if crop is not None:
                    cv2.imwrite(output_path, crop)
                    valid_paths.append(output_path)
                else:
                    valid_paths.append(None)
            except:
                valid_paths.append(None)
                
            if i % 500 == 0: gc.collect()

    df['segmented_path'] = valid_paths
    df_clean = df.dropna(subset=['segmented_path'])
    # Add dummy subject name for compatibility
    df_clean['subject_name'] = df_clean['filename'] 
    
    out_csv = os.path.join(FACET_OUTPUT_DIR, 'facet_final.csv')
    df_clean.to_csv(out_csv, index=False)
    print(f"FACET Done. Saved {len(df_clean)} images.")
    return out_csv

In [4]:
mste_out_csv_path = process_mste()


--- Processing MST-E Dataset ---


100%|██████████| 1546/1546 [03:42<00:00,  6.96it/s]

MST-E Done. Saved 1388 images.


In [21]:
facet_out_csv_path = process_facet(consensus_threshold=0.2)


--- Processing FACET Dataset ---
Consensus Filter: Dropped 189 images with low annotator agreement (<20.0%).


100%|██████████| 40211/40211 [22:36<00:00, 29.64it/s]

FACET Done. Saved 2867 images.



C:\Users\User\AppData\Local\Temp\ipykernel_82008\1354655381.py:183: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['subject_name'] = df_clean['filename']


In [14]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# ==========================================
# 1. CONFIGURATION
# ==========================================
MST_E_CSV = r'G:\\Thesis\\MonkSkinTone_Dataset\\Segmented_MSTE\\mst_e_final.csv'
FACET_CSV = r'G:\\Thesis\\FACET_Dataset\\Segmented_FACET\\facet_final.csv'

BATCH_SIZE = 32
NUM_EPOCHS = 25  # Increased since we have early stopping now
LEARNING_RATE = 1e-4
PATIENCE = 5     # Early stopping patience
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Running on Device: {DEVICE}")

# ==========================================
# 2. CUSTOM TRANSFORM (SHADES OF GRAY)
# ==========================================
class ShadesOfGray(object):
    def __init__(self, power=6):
        self.power = power

    def __call__(self, img):
        img_np = np.array(img)
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        mask = gray > 1
        
        if np.sum(mask) == 0: return img 
        
        b = img_bgr[:,:,0][mask].astype(np.float32)
        g = img_bgr[:,:,1][mask].astype(np.float32)
        r = img_bgr[:,:,2][mask].astype(np.float32)
        
        r_norm = np.power(np.mean(np.power(r, self.power)), 1/self.power)
        g_norm = np.power(np.mean(np.power(g, self.power)), 1/self.power)
        b_norm = np.power(np.mean(np.power(b, self.power)), 1/self.power)
        
        norm_vec = np.sqrt(r_norm**2 + g_norm**2 + b_norm**2)
        if norm_vec == 0: return img
        
        scale_r = 1.0 / (r_norm + 1e-6); scale_g = 1.0 / (g_norm + 1e-6); scale_b = 1.0 / (b_norm + 1e-6)
        max_scale = max(scale_r, scale_g, scale_b)
        scale_r /= max_scale; scale_g /= max_scale; scale_b /= max_scale
        
        result = img_bgr.astype(np.float32)
        result[:,:,0] *= (scale_b * 255)
        result[:,:,1] *= (scale_g * 255)
        result[:,:,2] *= (scale_r * 255)
        
        result = np.clip(result, 0, 255).astype(np.uint8)
        return Image.fromarray(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))

# ==========================================
# 3. DATASET & LOADERS
# ==========================================
class SkinToneDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['segmented_path']
        label = int(row['mst_label']) - 1 
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

def get_dataloaders():
    print("Loading Datasets...")
    df_mste = pd.read_csv(MST_E_CSV)
    df_facet = pd.read_csv(FACET_CSV)
    
    # ==========================================
    # 1. MST-E SPLIT (Stratified Subject Split)
    # ==========================================
    # Goal: Ensure EVERY skin tone class has representation in Train.
    # If a class has only 1 subject, they MUST go to Train.
    
    train_subjs_set = set()
    test_subjs_set = set()
    
    # Group data by Skin Tone Label to handle each class individually
    unique_labels = sorted(df_mste['mst_label'].unique())
    
    for label in unique_labels:
        # Get all unique subjects that belong to this specific skin tone
        # (Note: Some subjects might have images in multiple classes due to lighting, 
        # but we group by the label assigned to the image to ensure coverage)
        # Better approach for MST-E: Get subjects associated with this label 'mode' or just list them.
        # Since MST-E is structured by subject folders, let's look at subjects per class.
        
        # Get subset of dataframe for this label
        label_df = df_mste[df_mste['mst_label'] == label]
        subjects_in_label = label_df['subject_name'].unique()
        
        # Shuffle to ensure randomness
        np.random.seed(42)
        np.random.shuffle(subjects_in_label)
        
        n_subs = len(subjects_in_label)
        
        if n_subs == 1:
            # CRITICAL: Only 1 subject? Force to Train so model learns the class.
            train_subjs_set.update(subjects_in_label)
        else:
            # Standard Split: 80% Train, 20% Test
            # We want roughly 20% in Test, but prevent Test from being empty if possible
            n_test = max(1, int(n_subs * 0.2)) 
            
            # If 2 subjects, 1 train / 1 test is fine. 
            # If 10 subjects, 8 train / 2 test.
            
            # Assign
            test_subset = subjects_in_label[:n_test]
            train_subset = subjects_in_label[n_test:]
            
            test_subjs_set.update(test_subset)
            train_subjs_set.update(train_subset)
            
    # Resolve overlaps: If a subject appears in both sets (rare in MST-E, but possible),
    # prioritize Train to ensure training stability.
    test_subjs_set = test_subjs_set - train_subjs_set
    
    # Create DataFrames
    df_mste_train = df_mste[df_mste['subject_name'].isin(train_subjs_set)]
    df_mste_test = df_mste[df_mste['subject_name'].isin(test_subjs_set)]
    
    # ==========================================
    # 2. FACET SPLIT (Validation Heavy)
    # ==========================================
    # Goal: Use 20% for Training (Booster), 80% for Validation (Rigorous Check)
    FACET_TRAIN_RATIO = 0.7 
    
    df_facet = df_facet.sample(frac=1, random_state=42).reset_index(drop=True)
    train_size = int(len(df_facet) * FACET_TRAIN_RATIO)
    
    df_facet_train = df_facet[:train_size]  # Small portion for training
    df_facet_val = df_facet[train_size:]    # Large portion for validation
    
    # ==========================================
    # 3. MERGE & TRANSFORMS
    # ==========================================
    # Merge MST-E Train + FACET Train
    df_train_final = pd.concat([df_mste_train, df_facet_train], axis=0).reset_index(drop=True)
    
    print("-" * 40)
    print(f"TRAIN Set (MST-E + {int(FACET_TRAIN_RATIO*100)}% FACET): {len(df_train_final)} images")
    print(f"VAL Set   ({int((1-FACET_TRAIN_RATIO)*100)}% FACET):     {len(df_facet_val)} images")
    print(f"TEST Set  (MST-E Unseen):        {len(df_mste_test)} images")
    print("-" * 40)
    
    train_trans = transforms.Compose([
        transforms.Resize((224, 224)),
        ShadesOfGray(power=6),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    val_trans = transforms.Compose([
        transforms.Resize((224, 224)),
        ShadesOfGray(power=6),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    train_loader = DataLoader(SkinToneDataset(df_train_final, train_trans), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(SkinToneDataset(df_facet_val, val_trans), batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(SkinToneDataset(df_mste_test, val_trans), batch_size=BATCH_SIZE, shuffle=False)
    
    return train_loader, val_loader, test_loader

# ==========================================
# 4. MODEL & LOSS
# ==========================================
class OrdinalCrossEntropyLoss(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.num_classes = num_classes
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        preds = torch.argmax(logits, dim=1).detach()
        dist = torch.abs(targets - preds).float()
        weight = 1.0 + (dist / self.num_classes)
        return (ce * weight).mean()

def get_model():
    model = models.densenet121(weights='IMAGENET1K_V1')
    for param in model.parameters(): param.requires_grad = False
    for param in model.features.denseblock4.parameters(): param.requires_grad = True
    for param in model.features.norm5.parameters(): param.requires_grad = True
    model.classifier = nn.Linear(model.classifier.in_features, 10)
    return model.to(DEVICE)

# ==========================================
# 5. MAIN LOOP (With Early Stopping & Scheduler)
# ==========================================
def run_training():
    train_loader, val_loader, test_loader = get_dataloaders()
    model = get_model()
    criterion = OrdinalCrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
    
    # --- NEW: Scheduler ---
    # Reduces LR by factor of 0.1 if val_loss doesn't improve for 2 epochs
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2, verbose=True)
    
    # --- NEW: Tracking Variables ---
    best_val_loss = float('inf')
    epochs_no_improve = 0
    save_path = "best_model.pth"
    
    print("\n--- Starting Training ---")
    
    for epoch in range(NUM_EPOCHS):
        # 1. TRAIN
        model.train()
        train_loss = 0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
        
        for img, lbl in loop:
            img, lbl = img.to(DEVICE), lbl.to(DEVICE)
            optimizer.zero_grad()
            out = model(img)
            loss = criterion(out, lbl)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            loop.set_postfix(loss=loss.item())
            
        avg_train_loss = train_loss / len(train_loader)

        # 2. VALIDATE
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for img, lbl in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
                img, lbl = img.to(DEVICE), lbl.to(DEVICE)
                out = model(img)
                loss = criterion(out, lbl)
                val_loss += loss.item()
                
                _, pred = torch.max(out, 1)
                val_correct += (pred == lbl).sum().item()
                val_total += lbl.size(0)
        
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100 * val_correct / val_total
        
        print(f"Epoch {epoch+1}: Train Loss {avg_train_loss:.4f} | Val Loss {avg_val_loss:.4f} | Val Acc {val_acc:.2f}%")
        
        # 3. CHECKPOINTING & EARLY STOPPING
        scheduler.step(avg_val_loss) # Update LR based on Val Loss
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_path)
            print(f"--> New Best Model Saved! (Loss: {avg_val_loss:.4f})")
        else:
            epochs_no_improve += 1
            print(f"--> No improvement for {epochs_no_improve} epochs.")
            
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly Stopping triggered! No improvement for {PATIENCE} epochs.")
            break

    # 4. FINAL TEST (Load Best Model)
    print("\n--- Final Test on Unseen MST-E Subjects (Using Best Model) ---")
    model.load_state_dict(torch.load(save_path))
    model.eval()
    
    test_correct = 0
    test_total = 0
    diffs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for img, lbl in tqdm(test_loader, desc="Testing"):
            img, lbl = img.to(DEVICE), lbl.to(DEVICE)
            out = model(img)
            _, pred = torch.max(out, 1)
            
            test_correct += (pred == lbl).sum().item()
            test_total += lbl.size(0)
            diffs.extend(torch.abs(pred - lbl).cpu().numpy())
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(lbl.cpu().numpy())
            
    acc = 100 * test_correct / test_total
    oo_acc = 100 * np.mean(np.array(diffs) <= 1)
    
    print(f"Final Test Accuracy:       {acc:.2f}%")
    print(f"Final Off-by-One Accuracy: {oo_acc:.2f}%")
    
    from sklearn.metrics import classification_report
    print("\nDetailed Report:")
    print(classification_report(np.array(all_labels) + 1, np.array(all_preds) + 1, zero_division=0))

run_training()

Running on Device: cuda
Loading Datasets...
----------------------------------------
TRAIN Set (MST-E + 70% FACET): 1825 images
VAL Set   (30% FACET):     431 images
TEST Set  (MST-E Unseen):        567 images
----------------------------------------


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



--- Starting Training ---


Epoch 1 [Val]: 100%|██████████| 14/14 [00:03<00:00,  4.36it/s]


Epoch 1: Train Loss 2.6816 | Val Loss 2.3875 | Val Acc 19.72%
--> New Best Model Saved! (Loss: 2.3875)


...

Epoch 12: Train Loss 1.8562 | Val Loss 2.3270 | Val Acc 23.43%
--> No improvement for 4 epochs.


Epoch 13 [Val]: 100%|██████████| 14/14 [00:02<00:00,  4.81it/s]


Epoch 13: Train Loss 1.8355 | Val Loss 2.3509 | Val Acc 24.36%
--> No improvement for 5 epochs.

Early Stopping triggered! No improvement for 5 epochs.

--- Final Test on Unseen MST-E Subjects (Using Best Model) ---


Testing: 100%|██████████| 18/18 [00:10<00:00,  1.75it/s]

Final Test Accuracy:       12.70%
Final Off-by-One Accuracy: 34.39%

Detailed Report:
              precision    recall  f1-score   support

           1       0.00      0.00      0.00        82
           2       0.24      0.46      0.32        56
           3       0.00      0.00      0.00        38
           4       0.10      0.04      0.06        93
           5       0.22      0.44      0.29        73
           6       0.05      0.01      0.02        82
           7       0.00      0.00      0.00         0
           8       0.09      0.10      0.09        73
           9       0.50      0.03      0.05        70
          10       0.00      0.00      0.00         0

    accuracy                           0.13       567
   macro avg       0.12      0.11      0.08       567
weighted avg       0.15      0.13      0.10       567



In [22]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.utils.class_weight import compute_class_weight

# ==========================================
# 1. CONFIGURATION
# ==========================================
MST_E_CSV = r'G:\\Thesis\\MonkSkinTone_Dataset\\Segmented_MSTE\\mst_e_final.csv'
FACET_CSV = r'G:\\Thesis\\FACET_Dataset\\Segmented_FACET_0.2\\facet_final.csv'

BATCH_SIZE = 32
NUM_EPOCHS = 25
LEARNING_RATE = 1e-4
PATIENCE = 5
FACET_TRAIN_RATIO = 0.7 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. TRANSFORM
# ==========================================
class ShadesOfGray(object):
    def __init__(self, power=6):
        self.power = power
    def __call__(self, img):
        img_np = np.array(img)
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        mask = gray > 1
        if np.sum(mask) == 0: return img 
        
        b = img_bgr[:,:,0][mask].astype(np.float32)
        g = img_bgr[:,:,1][mask].astype(np.float32)
        r = img_bgr[:,:,2][mask].astype(np.float32)
        
        r_norm = np.power(np.mean(np.power(r, self.power)), 1/self.power)
        g_norm = np.power(np.mean(np.power(g, self.power)), 1/self.power)
        b_norm = np.power(np.mean(np.power(b, self.power)), 1/self.power)
        
        norm_vec = np.sqrt(r_norm**2 + g_norm**2 + b_norm**2)
        if norm_vec == 0: return img
        
        scale_r = 1.0 / (r_norm + 1e-6); scale_g = 1.0 / (g_norm + 1e-6); scale_b = 1.0 / (b_norm + 1e-6)
        max_scale = max(scale_r, scale_g, scale_b)
        scale_r /= max_scale; scale_g /= max_scale; scale_b /= max_scale
        
        result = img_bgr.astype(np.float32)
        result[:,:,0] *= (scale_b * 255)
        result[:,:,1] *= (scale_g * 255)
        result[:,:,2] *= (scale_r * 255)
        return Image.fromarray(cv2.cvtColor(np.clip(result, 0, 255).astype(np.uint8), cv2.COLOR_BGR2RGB))

# ==========================================
# 3. DATASET
# ==========================================
class SkinToneDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['segmented_path']
        label = int(row['mst_label']) - 1 
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

# ==========================================
# 4. LOADERS & WEIGHT CALCULATION
# ==========================================
def get_dataloaders():
    df_mste = pd.read_csv(MST_E_CSV)
    df_facet = pd.read_csv(FACET_CSV)
    
    # --- MST-E Split ---
    subjs = df_mste['subject_name'].unique()
    np.random.seed(42); np.random.shuffle(subjs)
    split_idx = int(len(subjs) * 0.8)
    train_subjs = subjs[:split_idx]
    test_subjs = subjs[split_idx:]
    
    df_mste_train = df_mste[df_mste['subject_name'].isin(train_subjs)]
    df_mste_test = df_mste[df_mste['subject_name'].isin(test_subjs)]
    
    # --- FACET Split ---
    df_facet = df_facet.sample(frac=1, random_state=42).reset_index(drop=True)
    train_size = int(len(df_facet) * FACET_TRAIN_RATIO)
    df_facet_train = df_facet[:train_size]
    df_facet_val = df_facet[train_size:]
    
    # --- Merge ---
    df_train_final = pd.concat([df_mste_train, df_facet_train], axis=0).reset_index(drop=True)
    
    print("-" * 40)
    print(f"TRAIN Set: {len(df_train_final)} images")
    print(f"VAL Set:   {len(df_facet_val)} images")
    print("-" * 40)
    
    # --- CALCULATE CLASS WEIGHTS ---
    # We compute weights based on the inverse frequency in the TRAINING set
    all_train_labels = df_train_final['mst_label'].values - 1 # 0-9 index
    class_weights = compute_class_weight('balanced', classes=np.unique(all_train_labels), y=all_train_labels)
    # Handle missing classes (if any)
    weights_tensor = torch.ones(10)
    for cls, weight in zip(np.unique(all_train_labels), class_weights):
        weights_tensor[cls] = weight
    print(f"Computed Class Weights: {weights_tensor}")

    # Transforms
    train_trans = transforms.Compose([
        transforms.Resize((224, 224)),
        # ShadesOfGray(power=6),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    val_trans = transforms.Compose([
        transforms.Resize((224, 224)),
        # ShadesOfGray(power=6),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    train_loader = DataLoader(SkinToneDataset(df_train_final, train_trans), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(SkinToneDataset(df_facet_val, val_trans), batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(SkinToneDataset(df_mste_test, val_trans), batch_size=BATCH_SIZE, shuffle=False)
    
    return train_loader, val_loader, test_loader, weights_tensor.to(DEVICE)

# ==========================================
# 5. MODEL & LOSS
# ==========================================
class WeightedOrdinalLoss(nn.Module):
    def __init__(self, num_classes=10, class_weights=None):
        super().__init__()
        self.num_classes = num_classes
        # Pass weights to CrossEntropy
        self.ce = nn.CrossEntropyLoss(weight=class_weights, reduction='none')

    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        preds = torch.argmax(logits, dim=1).detach()
        dist = torch.abs(targets - preds).float()
        weight = 1.0 + (dist / self.num_classes)
        return (ce * weight).mean()

def get_model():
    model = models.densenet121(weights='IMAGENET1K_V1')
    for param in model.parameters(): param.requires_grad = False
    for param in model.features.denseblock4.parameters(): param.requires_grad = True
    for param in model.features.norm5.parameters(): param.requires_grad = True
    
    # Add Dropout for Regularization
    num_ftrs = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.5), # Drop 50% of connections to prevent memorization
        nn.Linear(num_ftrs, 10)
    )
    return model.to(DEVICE)

# ==========================================
# 6. TRAINING
# ==========================================
def run_training():
    train_loader, val_loader, test_loader, class_weights = get_dataloaders()
    model = get_model()
    
    # Use Weighted Loss
    criterion = WeightedOrdinalLoss(num_classes=10, class_weights=class_weights)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2, verbose=True)
    
    best_val_loss = float('inf')
    save_path = "best_model.pth"
    
    print("\n--- Starting Training ---")
    for epoch in range(NUM_EPOCHS):
        model.train()
        train_loss = 0
        for img, lbl in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            img, lbl = img.to(DEVICE), lbl.to(DEVICE)
            optimizer.zero_grad()
            out = model(img)
            loss = criterion(out, lbl)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validate
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for img, lbl in val_loader:
                img, lbl = img.to(DEVICE), lbl.to(DEVICE)
                out = model(img)
                loss = criterion(out, lbl)
                val_loss += loss.item()
                _, pred = torch.max(out, 1)
                val_correct += (pred == lbl).sum().item()
                val_total += lbl.size(0)
        
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100 * val_correct / val_total
        print(f"Epoch {epoch+1}: Val Loss {avg_val_loss:.4f} | Val Acc {val_acc:.2f}%")
        
        scheduler.step(avg_val_loss)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), save_path)

    # Test
    print("\n--- Final Test ---")
    if os.path.exists(save_path): model.load_state_dict(torch.load(save_path))
    model.eval()
    
    test_correct = 0
    test_total = 0
    diffs = []
    all_preds, all_labels = [], []

    with torch.no_grad():
        for img, lbl in test_loader:
            img, lbl = img.to(DEVICE), lbl.to(DEVICE)
            out = model(img)
            _, pred = torch.max(out, 1)
            test_correct += (pred == lbl).sum().item()
            test_total += lbl.size(0)
            diffs.extend(torch.abs(pred - lbl).cpu().numpy())
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(lbl.cpu().numpy())
            
    acc = 100 * test_correct / test_total
    oo_acc = 100 * np.mean(np.array(diffs) <= 1)
    
    print(f"Test Acc: {acc:.2f}% | OOAcc: {oo_acc:.2f}%")
    from sklearn.metrics import classification_report
    print(classification_report(np.array(all_labels)+1, np.array(all_preds)+1, zero_division=0))

if __name__ == "__main__":
    run_training()

----------------------------------------
TRAIN Set: 3037 images
VAL Set:   861 images
----------------------------------------
Computed Class Weights: tensor([ 1.3498,  0.5655,  0.4914,  0.5522,  0.9148,  0.8436,  2.0383,  1.7354,
         3.5314, 60.7400])


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



--- Starting Training ---


Epoch 1: 100%|██████████| 95/95 [00:45<00:00,  2.10it/s]


Epoch 1: Val Loss 2.0148 | Val Acc 21.14%


...

Epoch 23: Val Loss 1.7974 | Val Acc 27.18%


Epoch 24: 100%|██████████| 95/95 [00:32<00:00,  2.91it/s]


Epoch 24: Val Loss 1.7948 | Val Acc 28.34%


Epoch 25: 100%|██████████| 95/95 [00:32<00:00,  2.88it/s]


Epoch 25: Val Loss 1.8141 | Val Acc 27.53%

--- Final Test ---
Test Acc: 11.48% | OOAcc: 58.26%
              precision    recall  f1-score   support

           1       0.00      0.00      0.00         0
           2       0.15      0.04      0.06        83
           3       0.00      0.00      0.00         0
           4       0.00      0.00      0.00         0
           5       0.90      0.10      0.17        94
           6       0.00      0.00      0.00         0
           7       0.00      0.00      0.00         0
           8       0.00      0.00      0.00         0
           9       0.58      0.40      0.47        70
          10       0.50      0.01      0.02       110

    accuracy                           0.11       357
   macro avg       0.21      0.05      0.07       357
weighted avg       0.54      0.11      0.16       357



In [23]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.utils.class_weight import compute_class_weight

# ==========================================
# 1. CONFIGURATION
# ==========================================
MST_E_CSV = r'G:\\Thesis\\MonkSkinTone_Dataset\\Segmented_MSTE\\mst_e_final.csv'
FACET_CSV = r'G:\\Thesis\\FACET_Dataset\\Segmented_FACET_0.2\\facet_final.csv'

BATCH_SIZE = 32
NUM_EPOCHS = 25
LEARNING_RATE = 1e-4
PATIENCE = 5
FACET_TRAIN_RATIO = 0.7
NUM_CLASSES = 3 # Changed from 10 to 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Running on Device: {DEVICE}")

# ==========================================
# 2. LABEL MAPPING HELPER
# ==========================================
def map_mst_to_group(mst_label):
    """
    Maps 10-scale MST to 3 groups:
    1-3 -> 0 (Light)
    4-7 -> 1 (Medium/Brown)
    8-10 -> 2 (Dark)
    """
    if mst_label <= 3:
        return 0
    elif mst_label <= 7:
        return 1
    else:
        return 2

GROUP_NAMES = ["Light (1-3)", "Medium (4-7)", "Dark (8-10)"]

# ==========================================
# 3. TRANSFORM (SHADES OF GRAY - OPTIONAL)
# ==========================================
class ShadesOfGray(object):
    def __init__(self, power=6):
        self.power = power
    def __call__(self, img):
        img_np = np.array(img)
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        mask = gray > 1
        if np.sum(mask) == 0: return img 
        
        b = img_bgr[:,:,0][mask].astype(np.float32)
        g = img_bgr[:,:,1][mask].astype(np.float32)
        r = img_bgr[:,:,2][mask].astype(np.float32)
        
        r_norm = np.power(np.mean(np.power(r, self.power)), 1/self.power)
        g_norm = np.power(np.mean(np.power(g, self.power)), 1/self.power)
        b_norm = np.power(np.mean(np.power(b, self.power)), 1/self.power)
        
        norm_vec = np.sqrt(r_norm**2 + g_norm**2 + b_norm**2)
        if norm_vec == 0: return img
        
        scale_r = 1.0 / (r_norm + 1e-6); scale_g = 1.0 / (g_norm + 1e-6); scale_b = 1.0 / (b_norm + 1e-6)
        max_scale = max(scale_r, scale_g, scale_b)
        scale_r /= max_scale; scale_g /= max_scale; scale_b /= max_scale
        
        result = img_bgr.astype(np.float32)
        result[:,:,0] *= (scale_b * 255)
        result[:,:,1] *= (scale_g * 255)
        result[:,:,2] *= (scale_r * 255)
        return Image.fromarray(cv2.cvtColor(np.clip(result, 0, 255).astype(np.uint8), cv2.COLOR_BGR2RGB))

# ==========================================
# 4. DATASET
# ==========================================
class SkinToneDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['segmented_path']
        
        # --- MAP LABEL HERE ---
        raw_mst = int(row['mst_label'])
        label = map_mst_to_group(raw_mst) # Returns 0, 1, or 2
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

# ==========================================
# 5. LOADERS & WEIGHT CALCULATION
# ==========================================
def get_dataloaders():
    df_mste = pd.read_csv(MST_E_CSV)
    df_facet = pd.read_csv(FACET_CSV)
    
    # --- MST-E Split ---
    subjs = df_mste['subject_name'].unique()
    np.random.seed(42); np.random.shuffle(subjs)
    split_idx = int(len(subjs) * 0.8)
    train_subjs = subjs[:split_idx]
    test_subjs = subjs[split_idx:]
    
    df_mste_train = df_mste[df_mste['subject_name'].isin(train_subjs)]
    df_mste_test = df_mste[df_mste['subject_name'].isin(test_subjs)]
    
    # --- FACET Split ---
    df_facet = df_facet.sample(frac=1, random_state=42).reset_index(drop=True)
    train_size = int(len(df_facet) * FACET_TRAIN_RATIO)
    df_facet_train = df_facet[:train_size]
    df_facet_val = df_facet[train_size:]
    
    # --- Merge ---
    df_train_final = pd.concat([df_mste_train, df_facet_train], axis=0).reset_index(drop=True)
    
    print("-" * 40)
    print(f"TRAIN Set: {len(df_train_final)} images")
    print(f"VAL Set:   {len(df_facet_val)} images")
    print("-" * 40)
    
    # --- CALCULATE CLASS WEIGHTS (FOR 3 GROUPS) ---
    # 1. Get raw labels
    raw_labels = df_train_final['mst_label'].values
    # 2. Map to groups
    grouped_labels = np.array([map_mst_to_group(l) for l in raw_labels])
    
    # 3. Compute weights for classes 0, 1, 2
    class_weights = compute_class_weight('balanced', classes=np.unique(grouped_labels), y=grouped_labels)
    
    weights_tensor = torch.tensor(class_weights, dtype=torch.float)
    print(f"Computed Group Weights (Light, Med, Dark): {weights_tensor}")

    # Transforms (ShadesOfGray DISABLED as per your previous snippet)
    train_trans = transforms.Compose([
        transforms.Resize((224, 224)),
        # ShadesOfGray(power=6), 
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    val_trans = transforms.Compose([
        transforms.Resize((224, 224)),
        # ShadesOfGray(power=6),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    train_loader = DataLoader(SkinToneDataset(df_train_final, train_trans), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(SkinToneDataset(df_facet_val, val_trans), batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(SkinToneDataset(df_mste_test, val_trans), batch_size=BATCH_SIZE, shuffle=False)
    
    return train_loader, val_loader, test_loader, weights_tensor.to(DEVICE)

# ==========================================
# 6. MODEL & LOSS
# ==========================================
class WeightedOrdinalLoss(nn.Module):
    def __init__(self, num_classes=3, class_weights=None): # Updated default to 3
        super().__init__()
        self.num_classes = num_classes
        self.ce = nn.CrossEntropyLoss(weight=class_weights, reduction='none')

    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        preds = torch.argmax(logits, dim=1).detach()
        dist = torch.abs(targets - preds).float()
        # Distance is smaller now (max 2), but logic still holds
        weight = 1.0 + (dist / self.num_classes)
        return (ce * weight).mean()

def get_model():
    model = models.densenet121(weights='IMAGENET1K_V1')
    for param in model.parameters(): param.requires_grad = False
    for param in model.features.denseblock4.parameters(): param.requires_grad = True
    for param in model.features.norm5.parameters(): param.requires_grad = True
    
    num_ftrs = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, NUM_CLASSES) # Outputs 3 classes
    )
    return model.to(DEVICE)

# ==========================================
# 7. TRAINING
# ==========================================
def run_training():
    train_loader, val_loader, test_loader, class_weights = get_dataloaders()
    model = get_model()
    
    criterion = WeightedOrdinalLoss(num_classes=NUM_CLASSES, class_weights=class_weights)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2, verbose=True)
    
    best_val_loss = float('inf')
    save_path = "best_model_grouped.pth"
    
    print("\n--- Starting Training (3 Groups) ---")
    for epoch in range(NUM_EPOCHS):
        model.train()
        train_loss = 0
        for img, lbl in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            img, lbl = img.to(DEVICE), lbl.to(DEVICE)
            optimizer.zero_grad()
            out = model(img)
            loss = criterion(out, lbl)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validate
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for img, lbl in val_loader:
                img, lbl = img.to(DEVICE), lbl.to(DEVICE)
                out = model(img)
                loss = criterion(out, lbl)
                val_loss += loss.item()
                _, pred = torch.max(out, 1)
                val_correct += (pred == lbl).sum().item()
                val_total += lbl.size(0)
        
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100 * val_correct / val_total
        print(f"Epoch {epoch+1}: Val Loss {avg_val_loss:.4f} | Val Acc {val_acc:.2f}%")
        
        scheduler.step(avg_val_loss)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), save_path)

    # Test
    print("\n--- Final Test on Unseen Subjects (3 Groups) ---")
    if os.path.exists(save_path): model.load_state_dict(torch.load(save_path))
    model.eval()
    
    test_correct = 0
    test_total = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for img, lbl in test_loader:
            img, lbl = img.to(DEVICE), lbl.to(DEVICE)
            out = model(img)
            _, pred = torch.max(out, 1)
            test_correct += (pred == lbl).sum().item()
            test_total += lbl.size(0)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(lbl.cpu().numpy())
            
    acc = 100 * test_correct / test_total
    
    print(f"Test Acc: {acc:.2f}%")
    
    from sklearn.metrics import classification_report
    # Map 0,1,2 back to names for the report
    print(classification_report(all_labels, all_preds, target_names=GROUP_NAMES, zero_division=0))

if __name__ == "__main__":
    run_training()

Running on Device: cuda
----------------------------------------
TRAIN Set: 3037 images
VAL Set:   861 images
----------------------------------------
Computed Group Weights (Light, Med, Dark): tensor([0.7336, 0.7278, 3.8058])


c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



--- Starting Training (3 Groups) ---


Epoch 1: 100%|██████████| 95/95 [00:32<00:00,  2.90it/s]


Epoch 1: Val Loss 0.8616 | Val Acc 59.23%


...

Epoch 23: Val Loss 0.8173 | Val Acc 66.55%


Epoch 24: 100%|██████████| 95/95 [00:32<00:00,  2.92it/s]


Epoch 24: Val Loss 0.8155 | Val Acc 67.25%


Epoch 25: 100%|██████████| 95/95 [00:32<00:00,  2.90it/s]


Epoch 25: Val Loss 0.8152 | Val Acc 66.90%

--- Final Test on Unseen Subjects (3 Groups) ---
Test Acc: 78.71%
              precision    recall  f1-score   support

 Light (1-3)       0.58      0.46      0.51        83
Medium (4-7)       0.58      0.68      0.63        94
 Dark (8-10)       0.98      0.99      0.99       180

    accuracy                           0.79       357
   macro avg       0.72      0.71      0.71       357
weighted avg       0.79      0.79      0.78       357



In [2]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Paths to your PROCESSED CSVs from the previous steps
MST_E_CSV = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\MonkSkinToneDataset\mst-e_data\mst_e_processed_robust.csv'
FACET_CSV = r'' #G:\Thesis\FACET_Dataset\facet_processed.csv' # Update if named differently

# Hyperparameters
BATCH_SIZE = 32
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
NUM_CLASSES = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Running on Device: {DEVICE}")

# ==========================================
# 2. CUSTOM DATASET CLASS
# ==========================================
class SkinToneDataset(Dataset):
    def __init__(self, df, transform=None):
        """
        df: DataFrame containing 'segmented_path' and 'mst_label'
        transform: PyTorch transforms (Augmentation/Normalization)
        """
        self.df = df
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['segmented_path']
        
        # Labels in CSV are likely 1-10. PyTorch needs 0-9.
        label = int(row['mst_label']) - 1 
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            # Fallback for corrupted images (return a black image)
            print(f"Error loading {img_path}: {e}")
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

# ==========================================
# 3. DATA TRANSFORMS (AUGMENTATION)
# ==========================================
# Training: Heavy Augmentation to learn invariance
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Validation/Test: No Augmentation (Standardize only)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ==========================================
# 4. DATA PREPARATION & SPLITTING
# ==========================================
def prepare_dataloaders():
    print("Loading Datasets...")
    
    # 1. Load MST-E (Gold Standard)
    if not os.path.exists(MST_E_CSV):
        raise FileNotFoundError(f"Could not find MST-E CSV at {MST_E_CSV}")
    
    df_mste = pd.read_csv(MST_E_CSV)
    
    # 2. Strict Subject Split on MST-E
    print("Performing Strict Subject Split on MST-E...")
    split_df = pd.DataFrame({
        'idx': df_mste.index, 
        'label': df_mste['mst_label'], 
        'subject': df_mste['subject_name']
    })
    
    train_idxs, test_idxs = [], []
    for label in sorted(split_df['label'].unique()):
        subjs = split_df[split_df['label'] == label]['subject'].unique()
        np.random.seed(42); np.random.shuffle(subjs)
        
        if len(subjs) == 1:
            train_subjs, test_subjs = subjs, []
        else:
            n_test = max(1, int(len(subjs) * 0.2))
            test_subjs = subjs[:n_test]
            train_subjs = subjs[n_test:]
            
        train_idxs.extend(split_df[split_df['subject'].isin(train_subjs)]['idx'].values)
        test_idxs.extend(split_df[split_df['subject'].isin(test_subjs)]['idx'].values)
        
    df_mste_train = df_mste.loc[train_idxs]
    df_mste_test = df_mste.loc[test_idxs]
    
    # 3. Load FACET (Booster) and add ONLY to TRAIN
    if os.path.exists(FACET_CSV):
        print("Merging FACET data into Training Set...")
        df_facet = pd.read_csv(FACET_CSV)
        # Combine
        df_train_final = pd.concat([df_mste_train, df_facet], axis=0).reset_index(drop=True)
    else:
        print("WARNING: FACET CSV not found. Training on small MST-E dataset only.")
        df_train_final = df_mste_train
    
    print(f"Final Train Size: {len(df_train_final)} images")
    print(f"Final Test Size:  {len(df_mste_test)} images (MST-E Unseen Subjects)")
    
    # 4. Create DataLoaders
    train_dataset = SkinToneDataset(df_train_final, transform=train_transforms)
    test_dataset = SkinToneDataset(df_mste_test, transform=val_transforms)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    return train_loader, test_loader

# ==========================================
# 5. MODEL ARCHITECTURE (DenseNet121)
# ==========================================
def get_model():
    print("Initializing DenseNet121...")
    model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    
    # 1. Freeze Initial Layers (Transfer Learning)
    for param in model.parameters():
        param.requires_grad = False
        
    # 2. Unfreeze 4th Dense Block (As per Paper)
    # In torchvision's DenseNet, 'features' contains the blocks.
    # denseblock4 is usually the last large block before norm5
    for param in model.features.denseblock4.parameters():
        param.requires_grad = True
    for param in model.features.norm5.parameters():
        param.requires_grad = True
        
    # 3. Replace Classifier
    num_ftrs = model.classifier.in_features
    model.classifier = nn.Linear(num_ftrs, NUM_CLASSES)
    
    return model.to(DEVICE)

# ==========================================
# 6. LOSS FUNCTION (Ordinal Cross Entropy)
# ==========================================
class OrdinalCrossEntropyLoss(nn.Module):
    """
    Paper Formula: OCE = (1 + |y - y_hat| / k) * CE
    Penalizes predictions further away from the true class.
    """
    def __init__(self, num_classes=10):
        super().__init__()
        self.num_classes = num_classes
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, logits, targets):
        # 1. Standard Cross Entropy
        ce_loss = self.ce(logits, targets)
        
        # 2. Get Predicted Class (for weighting)
        # We use .detach() because we don't want to backprop through the weight calculation itself
        preds = torch.argmax(logits, dim=1).detach()
        
        # 3. Calculate Weight: 1 + distance / k
        distance = torch.abs(targets - preds).float()
        weight = 1.0 + (distance / self.num_classes)
        
        # 4. Weighted Loss
        loss = ce_loss * weight
        return loss.mean()

# ==========================================
# 7. TRAINING & EVALUATION LOOP
# ==========================================
def train_and_evaluate():
    train_loader, test_loader = prepare_dataloaders()
    model = get_model()
    
    criterion = OrdinalCrossEntropyLoss(num_classes=NUM_CLASSES)
    # Only optimize parameters that require gradients (Block 4 + Classifier)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
    
    print("\nStarting Training...")
    
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
        
        for images, labels in loop:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            loop.set_postfix(loss=loss.item())
            
        train_acc = 100 * correct / total
        print(f"Epoch {epoch+1} Results -> Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}%")
        
        # --- Evaluate Every Epoch (Optional) ---
        evaluate(model, test_loader)

    print("\nTraining Complete. Final Evaluation on Unseen Subjects...")
    evaluate(model, test_loader)

def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Metrics
    from sklearn.metrics import accuracy_score, classification_report
    
    acc = accuracy_score(all_labels, all_preds)
    # Off-by-one
    diff = np.abs(all_labels - all_preds)
    oo_acc = np.mean(diff <= 1)
    
    print(f"\nTest Accuracy:       {acc:.2%}")
    print(f"Off-by-One Accuracy: {oo_acc:.2%}")
    print("\nDetailed Report:")
    # We add 1 to labels to match MST 1-10 scale in report
    print(classification_report(all_labels + 1, all_preds + 1, zero_division=0))

if __name__ == "__main__":
    train_and_evaluate()

Running on Device: cuda
Loading Datasets...
Performing Strict Subject Split on MST-E...
Final Train Size: 821 images
Final Test Size:  567 images (MST-E Unseen Subjects)
Initializing DenseNet121...

Starting Training...


Epoch 1/15: 100%|██████████| 26/26 [00:21<00:00,  1.21it/s, loss=1.62]


Epoch 1 Results -> Loss: 2.2507 | Train Acc: 43.61%


Evaluating: 100%|██████████| 18/18 [00:11<00:00,  1.62it/s]



Test Accuracy:       13.93%
Off-by-One Accuracy: 40.92%

Detailed Report:
              precision    recall  f1-score   support

           1       0.35      0.10      0.15        82
           2       0.24      0.84      0.37        56
           3       0.00      0.00      0.00        38
           4       0.10      0.12      0.11        93
           5       0.10      0.18      0.13        73
           6       0.00      0.00      0.00        82
           8       0.00      0.00      0.00        73
           9       0.00      0.00      0.00        70
          10       0.00      0.00      0.00         0

    accuracy                           0.14       567
   macro avg       0.09      0.14      0.08       567
weighted avg       0.10      0.14      0.09       567



...

Epoch 15 Results -> Loss: 0.0383 | Train Acc: 100.00%


Evaluating: 100%|██████████| 18/18 [00:10<00:00,  1.74it/s]



Test Accuracy:       34.22%
Off-by-One Accuracy: 62.79%

Detailed Report:
              precision    recall  f1-score   support

           1       0.82      0.28      0.42        82
           2       0.62      0.84      0.71        56
           3       0.00      0.00      0.00        38
           4       0.26      0.65      0.37        93
           5       0.00      0.00      0.00        73
           6       0.13      0.02      0.04        82
           7       0.00      0.00      0.00         0
           8       0.33      0.60      0.42        73
           9       0.90      0.26      0.40        70
          10       0.00      0.00      0.00         0

    accuracy                           0.34       567
   macro avg       0.31      0.26      0.24       567
weighted avg       0.40      0.34      0.30       567


Training Complete. Final Evaluation on Unseen Subjects...


Evaluating: 100%|██████████| 18/18 [00:10<00:00,  1.74it/s]


Test Accuracy:       34.22%
Off-by-One Accuracy: 62.79%

Detailed Report:
              precision    recall  f1-score   support

           1       0.82      0.28      0.42        82
           2       0.62      0.84      0.71        56
           3       0.00      0.00      0.00        38
           4       0.26      0.65      0.37        93
           5       0.00      0.00      0.00        73
           6       0.13      0.02      0.04        82
           7       0.00      0.00      0.00         0
           8       0.33      0.60      0.42        73
           9       0.90      0.26      0.40        70
          10       0.00      0.00      0.00         0

    accuracy                           0.34       567
   macro avg       0.31      0.26      0.24       567
weighted avg       0.40      0.34      0.30       567



In [ ]:
# Running on Device: cuda
# Loading Datasets...
# Performing Strict Subject Split on MST-E...
# WARNING: FACET CSV not found. Training on small MST-E dataset only.
# Final Train Size: 821 images
# Final Test Size:  567 images (MST-E Unseen Subjects)
# Initializing DenseNet121...
# Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to C:\Users\User/.cache\torch\hub\checkpoints\densenet121-a639ec97.pth
# 100%|██████████| 30.8M/30.8M [00:01<00:00, 22.6MB/s]

# Starting Training...
# Epoch 1/15: 100%|██████████| 26/26 [00:23<00:00,  1.10it/s, loss=1.68]
# Epoch 1 Results -> Loss: 2.2668 | Train Acc: 43.61%
# Epoch 2/15: 100%|██████████| 26/26 [00:22<00:00,  1.13it/s, loss=1.07] 
# Epoch 2 Results -> Loss: 1.1916 | Train Acc: 83.56%
# Epoch 3/15: 100%|██████████| 26/26 [00:21<00:00,  1.19it/s, loss=0.613]
# Epoch 3 Results -> Loss: 0.7126 | Train Acc: 92.69%
# Epoch 4/15: 100%|██████████| 26/26 [00:21<00:00,  1.22it/s, loss=0.542]
# Epoch 4 Results -> Loss: 0.4999 | Train Acc: 96.35%
# Epoch 5/15: 100%|██████████| 26/26 [00:22<00:00,  1.17it/s, loss=0.274]
# Epoch 5 Results -> Loss: 0.3358 | Train Acc: 97.44%
# Epoch 6/15: 100%|██████████| 26/26 [00:23<00:00,  1.12it/s, loss=0.31] 
# Epoch 6 Results -> Loss: 0.2480 | Train Acc: 98.66%
# Epoch 7/15: 100%|██████████| 26/26 [00:22<00:00,  1.16it/s, loss=0.145]
# Epoch 7 Results -> Loss: 0.1766 | Train Acc: 99.27%
# Epoch 8/15: 100%|██████████| 26/26 [00:21<00:00,  1.20it/s, loss=0.0784]
# Epoch 8 Results -> Loss: 0.1424 | Train Acc: 99.03%
# Epoch 9/15: 100%|██████████| 26/26 [00:22<00:00,  1.18it/s, loss=0.137] 
# Epoch 9 Results -> Loss: 0.1197 | Train Acc: 99.27%
# Epoch 10/15: 100%|██████████| 26/26 [00:21<00:00,  1.19it/s, loss=0.147] 
# Epoch 10 Results -> Loss: 0.0970 | Train Acc: 99.63%
# Epoch 11/15: 100%|██████████| 26/26 [00:22<00:00,  1.18it/s, loss=0.144] 
# Epoch 11 Results -> Loss: 0.0823 | Train Acc: 99.51%
# Epoch 12/15: 100%|██████████| 26/26 [00:22<00:00,  1.18it/s, loss=0.107] 
# Epoch 12 Results -> Loss: 0.0663 | Train Acc: 99.76%
# Epoch 13/15: 100%|██████████| 26/26 [00:21<00:00,  1.20it/s, loss=0.0434]
# Epoch 13 Results -> Loss: 0.0556 | Train Acc: 99.76%
# Epoch 14/15: 100%|██████████| 26/26 [00:20<00:00,  1.24it/s, loss=0.0392]
# Epoch 14 Results -> Loss: 0.0477 | Train Acc: 99.88%
# Epoch 15/15: 100%|██████████| 26/26 [00:21<00:00,  1.23it/s, loss=0.0634]
# Epoch 15 Results -> Loss: 0.0417 | Train Acc: 100.00%

# Training Complete. Final Evaluation on Unseen Subjects...
# Evaluating: 100%|██████████| 18/18 [00:11<00:00,  1.56it/s]

# Test Accuracy:       37.04%
# Off-by-One Accuracy: 64.55%

# Detailed Report:
#               precision    recall  f1-score   support

#            1       0.80      0.43      0.56        82
#            2       0.55      0.91      0.69        56
#            3       0.00      0.00      0.00        38
#            4       0.25      0.56      0.34        93
#            5       0.12      0.01      0.02        73
#            6       0.09      0.01      0.02        82
#            7       0.00      0.00      0.00         0
#            8       0.40      0.70      0.50        73
#            9       0.90      0.27      0.42        70
#           10       0.00      0.00      0.00         0

#     accuracy                           0.37       567
#    macro avg       0.31      0.29      0.26       567
# weighted avg       0.40      0.37      0.33       567